<a href="https://colab.research.google.com/github/usama488/bioinformatics-analysis/blob/main/Bioinformatics_Dashboard.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#  Bioinformatics Dashboard

# Project 6 Bioinformatics

Yeh ek integrated interactive dashboard hai jo pichlay projects (clinical data, gene expression, variants) ko ek jaga par la kar explore karne deta hai — sab kuch tabs mein organized, Plotly ke sath fully interactive.

**Dashboard Tabs:**
1.  **Overview** — dataset summary, class distribution
2.  **Gene Expression Explorer** — koi bhi gene select kar ke uski expression pattern dekhein
3.  **Clinical Data Explorer** — patient clinical variables ka interactive analysis
4.  **Variant Summary** — mutation burden aur variant spectrum
5.  **Risk Prediction** — apna patient data daal kar direct risk/diagnosis predict karein (runtime cell)

> Poora dashboard ek jaga par — koi bhi cell independently bhi run ho sakta hai.


## 1. Setup & Imports

In [1]:
# !pip install -q plotly scikit-learn pandas numpy ipywidgets

import numpy as np
import pandas as pd

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report, confusion_matrix

import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

np.random.seed(42)


## 2. Build Combined Patient Dataset

**Real data use karne ke liye:** apni clinical CSV + expression matrix ko patient_df mein merge kar lein (same structure follow karein).


In [2]:
n_patients = 250
rng = np.random.default_rng(42)

diagnosis = rng.choice(["Cancer", "No Cancer"], size=n_patients, p=[0.45, 0.55])

age = np.clip(rng.normal(58, 12, n_patients), 20, 90).round(0)
tumor_size_mm = np.where(diagnosis == "Cancer", rng.normal(28, 10, n_patients), rng.normal(5, 3, n_patients)).clip(0)
mutation_burden = np.where(diagnosis == "Cancer", rng.poisson(45, n_patients), rng.poisson(12, n_patients))
smoking_status = rng.choice(["Never", "Former", "Current"], size=n_patients, p=[0.5, 0.3, 0.2])
stage = np.where(diagnosis == "Cancer", rng.choice(["I", "II", "III", "IV"], n_patients, p=[0.3, 0.3, 0.25, 0.15]), "N/A")

gene_names = [f"GENE_{i}" for i in range(1, 11)]
gene_expr = {}
for i, gene in enumerate(gene_names):
    shift = rng.uniform(0.5, 2.5) if i % 2 == 0 else 0
    base = rng.normal(6, 1.2, n_patients)
    gene_expr[gene] = np.where(diagnosis == "Cancer", base + shift, base)

patient_df = pd.DataFrame({
    "patient_id": [f"P{str(i+1).zfill(4)}" for i in range(n_patients)],
    "age": age, "tumor_size_mm": tumor_size_mm.round(1), "mutation_burden": mutation_burden,
    "smoking_status": smoking_status, "stage": stage, "diagnosis": diagnosis,
    **{g: v.round(2) for g, v in gene_expr.items()}
})

print(f"Dataset shape: {patient_df.shape}")
patient_df.head()


Dataset shape: (250, 17)


,patient_id,age,tumor_size_mm,mutation_burden,smoking_status,stage,diagnosis,GENE_1,GENE_2,GENE_3,GENE_4,GENE_5,GENE_6,GENE_7,GENE_8,GENE_9,GENE_10
0,P0001,57.0,8.6,16,Former,N/A,No Cancer,2.57,6.14,7.23,5.73,5.67,5.01,6.04,6.46,5.33,6.75
1,P0002,37.0,25.1,50,Current,I,Cancer,8.54,6.22,6.39,6.93,8.37,7.57,8.74,5.94,9.40,6.31
2,P0003,40.0,0.0,16,Former,N/A,No Cancer,5.86,8.34,6.05,6.12,6.80,7.61,6.66,7.30,4.94,5.41
3,P0004,84.0,5.3,11,Former,N/A,No Cancer,5.94,4.48,5.93,4.30,5.82,6.62,5.84,5.26,5.30,5.22
4,P0005,43.0,37.4,43,Former,II,Cancer,9.03,5.97,7.90,4.42,7.90,5.45,8.24,7.18,10.04,6.15


## 3. Dashboard  Tab 1:  Overview

In [3]:
overview_out = widgets.Output()

with overview_out:
    n_cancer = (patient_df["diagnosis"] == "Cancer").sum()
    n_healthy = (patient_df["diagnosis"] == "No Cancer").sum()

    summary_html = f"""
    <div style="display:flex; gap:16px; font-family:sans-serif; margin-bottom:15px;">
        <div style="flex:1; background:#eef6f9; border-radius:10px; padding:16px; text-align:center;">
            <div style="font-size:26px; font-weight:700; color:#2C5364;">{len(patient_df)}</div>
            <div style="font-size:13px; color:#555;">Total Patients</div>
        </div>
        <div style="flex:1; background:#fdeceb; border-radius:10px; padding:16px; text-align:center;">
            <div style="font-size:26px; font-weight:700; color:#E63946;">{n_cancer}</div>
            <div style="font-size:13px; color:#555;">Cancer Cases</div>
        </div>
        <div style="flex:1; background:#e9f4fb; border-radius:10px; padding:16px; text-align:center;">
            <div style="font-size:26px; font-weight:700; color:#2E86AB;">{n_healthy}</div>
            <div style="font-size:13px; color:#555;">No Cancer</div>
        </div>
        <div style="flex:1; background:#f0f7f0; border-radius:10px; padding:16px; text-align:center;">
            <div style="font-size:26px; font-weight:700; color:#43AA8B;">{patient_df['age'].mean():.0f}</div>
            <div style="font-size:13px; color:#555;">Avg. Age</div>
        </div>
    </div>
    """
    display(HTML(summary_html))

    fig = make_subplots(rows=1, cols=2, subplot_titles=("Diagnosis Distribution", "Cancer Stage Distribution"),
                         specs=[[{"type": "domain"}, {"type": "domain"}]])
    diag_counts = patient_df["diagnosis"].value_counts()
    fig.add_trace(go.Pie(labels=diag_counts.index, values=diag_counts.values, hole=0.45,
                          marker=dict(colors=["#E63946", "#2E86AB"])), row=1, col=1)
    stage_counts = patient_df[patient_df["stage"] != "N/A"]["stage"].value_counts().sort_index()
    fig.add_trace(go.Pie(labels=stage_counts.index, values=stage_counts.values, hole=0.45,
                          marker=dict(colors=px.colors.sequential.Reds)), row=1, col=2)
    fig.update_layout(height=400, title_text="Cohort Overview")
    fig.show()

    fig2 = px.histogram(patient_df, x="age", color="diagnosis", barmode="overlay", nbins=25,
                         title="Age Distribution by Diagnosis", template="plotly_white",
                         color_discrete_map={"Cancer": "#E63946", "No Cancer": "#2E86AB"}, opacity=0.65)
    fig2.update_layout(height=400)
    fig2.show()

display(overview_out)


Output()

## 4. Dashboard — Tab 2:  Gene Expression Explorer

In [4]:
gene_dropdown = widgets.Dropdown(options=gene_names, value=gene_names[0], description="Select Gene:",
                                  style={'description_width': '100px'}, layout=widgets.Layout(width='300px'))
expr_out = widgets.Output()

def update_expr_view(change=None):
    with expr_out:
        clear_output()
        gene = gene_dropdown.value
        fig = make_subplots(rows=1, cols=2, subplot_titles=(f"{gene} — by Diagnosis", f"{gene} — by Smoking Status"))

        for i, (col, cats, cmap) in enumerate([
            ("diagnosis", ["Cancer", "No Cancer"], {"Cancer": "#E63946", "No Cancer": "#2E86AB"}),
            ("smoking_status", ["Never", "Former", "Current"], None)
        ]):
            for cat in cats:
                vals = patient_df.loc[patient_df[col] == cat, gene]
                color = cmap[cat] if cmap else None
                fig.add_trace(go.Box(y=vals, name=cat, marker_color=color, showlegend=False), row=1, col=i+1)

        fig.update_layout(height=450, title_text=f"Expression Explorer — {gene}")
        fig.show()

        fig2 = px.scatter(patient_df, x="mutation_burden", y=gene, color="diagnosis",
                           title=f"{gene} Expression vs Mutation Burden", template="plotly_white",
                           color_discrete_map={"Cancer": "#E63946", "No Cancer": "#2E86AB"})
        fig2.update_layout(height=450)
        fig2.show()

gene_dropdown.observe(update_expr_view, names='value')
display(gene_dropdown, expr_out)
update_expr_view()


Dropdown(description='Select Gene:', layout=Layout(width='300px'), options=('GENE_1', 'GENE_2', 'GENE_3', 'GEN…

Output()

## 5. Dashboard — Tab 3:  Clinical Data Explorer

In [8]:
clinical_out = widgets.Output()
with clinical_out:
    fig = px.scatter(patient_df, x="age", y="tumor_size_mm", color="diagnosis", size="mutation_burden",
                      hover_data=["patient_id", "stage"],
                      title="Age vs Tumor Size (bubble size = mutation burden)",
                      template="plotly_white", color_discrete_map={"Cancer": "#E63946", "No Cancer": "#2E86AB"})
    fig.update_layout(height=500)
    fig.show()

    fig2 = px.box(patient_df, x="smoking_status", y="mutation_burden", color="diagnosis",
                   title="Mutation Burden by Smoking Status", template="plotly_white",
                   color_discrete_map={"Cancer": "#E63946", "No Cancer": "#2E86AB"})
    fig2.update_layout(height=450)
    fig2.show()

    numeric_cols = ["age", "tumor_size_mm", "mutation_burden"] + gene_names
    corr = patient_df[numeric_cols].corr()
    fig3 = px.imshow(corr, color_continuous_scale="RdBu_r", aspect="auto",
                      title="Clinical & Expression Feature Correlation Matrix")
    fig3.update_layout(height=550)
    fig3.show()

display(clinical_out)

Output()

## 6. Dashboard — Tab 4:  Variant / Mutation Summary

In [9]:
variant_out = widgets.Output()
with variant_out:
    fig = px.violin(patient_df, x="diagnosis", y="mutation_burden", color="diagnosis", box=True, points="all",
                     title="Mutation Burden Distribution — Cancer vs No Cancer",
                     template="plotly_white", color_discrete_map={"Cancer": "#E63946", "No Cancer": "#2E86AB"})
    fig.update_layout(height=500)
    fig.show()

    stage_burden = patient_df[patient_df["stage"] != "N/A"].groupby("stage")["mutation_burden"].mean().reindex(["I", "II", "III", "IV"])
    fig2 = px.bar(stage_burden, title="Average Mutation Burden by Cancer Stage",
                   labels={"value": "Avg. Mutation Burden", "stage": "Stage"},
                   template="plotly_white", color=stage_burden.values, color_continuous_scale="Reds")
    fig2.update_layout(height=450, showlegend=False)
    fig2.show()

display(variant_out)


Output()

## 7. Full Dashboard — Tabbed View

In [10]:
tab = widgets.Tab(children=[overview_out, widgets.VBox([gene_dropdown, expr_out]), clinical_out, variant_out])
tab.set_title(0, " Overview")
tab.set_title(1, " Gene Expression")
tab.set_title(2, " Clinical Data")
tab.set_title(3, " Mutation Summary")
display(tab)


## 8. Train Risk Prediction Model

In [11]:
feature_cols = ["age", "tumor_size_mm", "mutation_burden"] + gene_names
X = patient_df[feature_cols].copy()
y = (patient_df["diagnosis"] == "Cancer").astype(int)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=42)

dash_scaler = StandardScaler()
X_train_scaled = dash_scaler.fit_transform(X_train)
X_test_scaled = dash_scaler.transform(X_test)

dash_clf = RandomForestClassifier(n_estimators=300, random_state=42)
dash_clf.fit(X_train_scaled, y_train)

preds = dash_clf.predict(X_test_scaled)
probs = dash_clf.predict_proba(X_test_scaled)[:, 1]
print(f"Accuracy: {accuracy_score(y_test, preds):.3f}  |  ROC-AUC: {roc_auc_score(y_test, probs):.3f}")
print("\n", classification_report(y_test, preds, target_names=["No Cancer", "Cancer"]))


Accuracy: 1.000  |  ROC-AUC: 1.000

               precision    recall  f1-score   support

   No Cancer       1.00      1.00      1.00        35
      Cancer       1.00      1.00      1.00        28

    accuracy                           1.00        63
   macro avg       1.00      1.00      1.00        63
weighted avg       1.00      1.00      1.00        63



## 9.  Dashboard Tab 5: Runtime Risk Prediction




In [12]:
age_box = widgets.IntSlider(value=55, min=20, max=90, description="Age", style={'description_width': '150px'}, layout=widgets.Layout(width='400px'))
tumor_box = widgets.FloatSlider(value=15.0, min=0, max=60, step=0.5, description="Tumor Size (mm)", style={'description_width': '150px'}, layout=widgets.Layout(width='400px'))
mutation_box = widgets.IntSlider(value=25, min=0, max=100, description="Mutation Burden", style={'description_width': '150px'}, layout=widgets.Layout(width='400px'))

gene_defaults_dash = patient_df[gene_names].mean().to_dict()
gene_boxes_dash = {}
gene_widget_list = []
for gene in gene_names:
    box = widgets.FloatText(value=round(gene_defaults_dash[gene], 2), description=gene,
                             style={'description_width': '90px'}, layout=widgets.Layout(width='220px'))
    gene_boxes_dash[gene] = box
    gene_widget_list.append(box)

gene_grid = widgets.GridBox(gene_widget_list, layout=widgets.Layout(grid_template_columns="repeat(5, 230px)", grid_gap="6px"))

predict_btn = widgets.Button(description=" Risk Predict Karein", button_style='success',
                              layout=widgets.Layout(width='240px', height='42px'))
out_final = widgets.Output()

def render_dash_result(label, proba):
    color = "#E63946" if label == "Cancer" else "#2E86AB"
    emoji = "" if label == "Cancer" else ""
    conf = proba[1]*100 if label == "Cancer" else proba[0]*100
    html = f"""
    <div style="border:2px solid {color}; border-radius:12px; padding:18px; margin-top:12px; font-family:sans-serif; background:#fafafa;">
        <div style="font-size:22px; font-weight:700; color:{color};">{emoji} Risk Prediction: {label}</div>
        <div style="font-size:15px; margin-top:8px;">Confidence: <b>{conf:.2f}%</b></div>
        <div style="margin-top:10px; height:14px; width:100%; background:#e0e0e0; border-radius:7px; overflow:hidden;">
            <div style="height:100%; width:{proba[1]*100:.1f}%; background:linear-gradient(90deg,#2E86AB,#E63946);"></div>
        </div>
        <div style="display:flex; justify-content:space-between; font-size:12px; color:#555; margin-top:4px;">
            <span>P(No Cancer) = {proba[0]*100:.2f}%</span><span>P(Cancer) = {proba[1]*100:.2f}%</span>
        </div>
    </div>
    """
    display(HTML(html))

def on_predict_dash(b):
    with out_final:
        clear_output()
        row = {"age": age_box.value, "tumor_size_mm": tumor_box.value, "mutation_burden": mutation_box.value}
        row.update({g: gene_boxes_dash[g].value for g in gene_names})
        row_df = pd.DataFrame([row])[feature_cols]
        row_scaled = dash_scaler.transform(row_df)
        pred = dash_clf.predict(row_scaled)[0]
        proba = dash_clf.predict_proba(row_scaled)[0]
        label = "Cancer" if pred == 1 else "No Cancer"
        render_dash_result(label, proba)

predict_btn.on_click(on_predict_dash)

display(widgets.HTML("<b style='font-size:15px;'>Clinical Info</b>"))
display(age_box, tumor_box, mutation_box)
display(widgets.HTML("<br><b style='font-size:15px;'>Gene Expression</b>"))
display(gene_grid)
display(predict_btn)
display(out_final)


HTML(value="<b style='font-size:15px;'>Clinical Info</b>")

IntSlider(value=55, description='Age', layout=Layout(width='400px'), max=90, min=20, style=SliderStyle(descrip…

FloatSlider(value=15.0, description='Tumor Size (mm)', layout=Layout(width='400px'), max=60.0, step=0.5, style…

IntSlider(value=25, description='Mutation Burden', layout=Layout(width='400px'), style=SliderStyle(description…

HTML(value="<br><b style='font-size:15px;'>Gene Expression</b>")

GridBox(children=(FloatText(value=6.98, description='GENE_1', layout=Layout(width='220px'), style=DescriptionS…

Button(button_style='success', description='🔮 Risk Predict Karein', layout=Layout(height='42px', width='240px'…

Output()